# Political Representation

Need a legistar token

## Pulling scores using API

Three dataframes are created:
* `bills_df` which contains information on all street-vending related bills
* `enacted_df` which contains information only for enacted bills
* `df_council_votes` which contains one row per vote of each council district for all enacted laws

In [ ]:
# Setup

import requests
import pandas as pd
import time
from dotenv import load_dotenv
import sys, os
sys.path.append(os.path.abspath("..")) 

load_dotenv()  # reads .env from the project root (or nearest parent dir)
TOKEN = os.environ.get("LEGISTAR_TOKEN")

if TOKEN is None:
    raise ValueError("LEGISTAR_TOKEN not found — check your .env file")
BASE = "https://webapi.legistar.com/v1/nyc"


FIGURES_DIR = "../outputs"
os.makedirs(FIGURES_DIR, exist_ok=True)

DATA_DIR = "../data/processed"

def get(endpoint, params={}):
    params["token"] = TOKEN
    r = requests.get(f"{BASE}{endpoint}", params=params)
    r.raise_for_status()
    return r.json()

In [ ]:
# Pull all bills with street vending keywords

keywords = [
    "street vend", "street vendor", "sidewalk vend",
    "mobile food", "food vendor", "general vendor",
    "vending license", "vending permit", "pushcart",
    "decriminalize vend", "vendor penalty", "vendor inspection",
    "vendor license", "vending cart", "food protection"
]

all_matters = []
for kw in keywords:
    skip = 0
    while True:
        results = get("/matters", {
            "$top": 1000,
            "$skip": skip,
            "$filter": f"substringof('{kw}', MatterTitle) eq true"
        })
        if not results:
            break
        all_matters.extend(results)
        skip += 1000
        time.sleep(0.3)

# Deduplicate by MatterId
matters_df = pd.DataFrame(all_matters).drop_duplicates("MatterId")

# Filter to bills (not resolutions, hearings, etc.) and since 1998
matters_df["MatterIntroDate"] = pd.to_datetime(matters_df["MatterIntroDate"], errors="coerce")
all_bills = matters_df[
    (matters_df["MatterTypeName"].str.contains("Introduction|Local Law|Bill", case=False, na=False)) &
    (matters_df["MatterIntroDate"].dt.year >= 1998) &
    (matters_df['MatterIntroDate'].dt.year < 2026)
].copy()

all_bills.to_csv(f"{DATA_DIR}/political_representation/all_bills.csv")

In [1]:
import requests
r = requests.get("https://webapi.legistar.com/v1/nyc/matters", params={"$top": 1})
print(r.status_code)

403
